# Amazon Review - OPTIMIZED PySpark Preprocessing

**Optimizations Applied:**
- Broadcast joins for metadata
- Repartitioning for better parallelism
- Caching strategic DataFrames
- Batch processing for numerical operations
- Optimized text cleaning with native Spark functions
- Predicate pushdown and column pruning

## 1. Setup Spark Session (OPTIMIZED)

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import multiprocessing

# Tự động đếm số lõi CPU thực tế trên máy bạn
cores = multiprocessing.cpu_count()
# Cấu hình số partition thường gấp 2-3 lần số lõi CPU để tối ưu hóa
safe_cores = max(4, int(cores * 0.6))
num_partitions = safe_cores * 3
# OPTIMIZED Spark Configuration
spark = SparkSession.builder \
    .appName("Amazon Review Local Processing") \
    .master(f"local[{safe_cores}]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "50MB") \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
    .config("spark.memory.fraction", "0.6") \
    .getOrCreate()
print(f"✅ Spark {spark.version} initialized for LOCAL MODE")
print(f"🖥️  CPU Cores utilized: {cores}")
print(f"📊 Partitions configured: {num_partitions}")
print(f"💾 Driver Memory (Max RAM): 8GB")

✅ Spark 3.5.1 initialized for LOCAL MODE
🖥️  CPU Cores utilized: 16
📊 Partitions configured: 27
💾 Driver Memory (Max RAM): 8GB


## 2. Configuration

In [2]:
from pathlib import Path

ROOT_DIR      = Path().resolve().parent
DATA_DIR      = ROOT_DIR / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Đảm bảo thư mục processed tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 📥 Đường dẫn đầu vào (Ưu tiên dùng Parquet đã convert để nhanh hơn)
REVIEW_RAW_PARQUET = str(RAW_DIR / "Clothing_Shoes_and_Jewelry.parquet")
META_RAW_PARQUET   = str(RAW_DIR / "meta_Clothing_Shoes_and_Jewelry.parquet")

# 📤 Đường dẫn đầu ra cho các bước xử lý tiếp theo
REVIEW_CLEAN_PARQUET = str(PROCESSED_DIR / "review_clean_spark.parquet")
META_CLEAN_PARQUET   = str(PROCESSED_DIR / "meta_clean_spark.parquet")
MERGED_PARQUET      = str(PROCESSED_DIR / "amazon_merged_spark.parquet")
FINAL_PARQUET       = str(PROCESSED_DIR / "amazon_final_gold_spark.parquet")

# Các tham số cấu hình khác
K_CORE = 5
OUTLIER_ZSCORE_THRESHOLD = 3.0
MIN_TEXT_LENGTH = 10
MAX_TEXT_LENGTH = 5000  
NUM_PARTITIONS = 200
print(f"📂 Project Root: {ROOT_DIR}")
print(f"✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.")

📂 Project Root: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis
✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.


## 3. Optimized Text Cleaning

In [ ]:
# ============================================================
# OPTIMIZED TEXT CLEANING - Single Pass for Both Variants
# ============================================================

STOPWORDS_LIST = [
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'he','him','his','she','her','hers','it','its','they','them','their',
    'what','which','who','this','that','these','those','am','is','are','was',
    'were','be','been','being','have','has','had','do','does','did','will',
    'would','could','should','may','might','shall','can','a','an','the',
    'and','but','or','nor','for','so','yet','both','either','neither',
    'not','no','nor','only','own','same','than','too','very','just',
    'because','as','until','while','of','at','by','with','about','against',
    'between','through','during','before','after','above','below','to',
    'from','in','out','on','off','over','under','again','then','once'
]

def apply_optimized_text_cleaning(df, text_col='combined_text'):
    """
    🚀 OPTIMIZED: Process both BERT and TF-IDF cleaning in ONE PASS
    This is 2x faster than calling two separate functions
    """
    print("🚀 Applying optimized text cleaning (single pass for both variants)...")
    
    # Get base text
    base_text = F.coalesce(F.col(text_col), F.lit(""))
    
    # ========================================
    # BERT CLEANING (semantic preserving)
    # ========================================
    bert_clean = base_text
    bert_clean = F.regexp_replace(bert_clean, r'<[^>]+>', ' ')  # HTML tags
    bert_clean = F.regexp_replace(bert_clean, r'http\S+|www\.\S+', ' ')  # URLs
    bert_clean = F.regexp_replace(bert_clean, r'&[a-z]+;', ' ')  # HTML entities
    bert_clean = F.regexp_replace(bert_clean, r'\s+', ' ')  # Whitespace
    bert_clean = F.substring(F.trim(bert_clean), 1, 512)  # Limit 512 chars
    
    # ========================================
    # TF-IDF CLEANING (aggressive)
    # ========================================
    tfidf_clean = F.lower(base_text)
    tfidf_clean = F.regexp_replace(tfidf_clean, r'<[^>]+>', ' ')  # HTML
    tfidf_clean = F.regexp_replace(tfidf_clean, r'[^a-z0-9\s]', ' ')  # Special chars
    
    # Remove stopwords - OPTIMIZED with single regex
    stopwords_pattern = r'\b(' + '|'.join(STOPWORDS_LIST) + r')\b'
    tfidf_clean = F.regexp_replace(tfidf_clean, stopwords_pattern, ' ')
    
    tfidf_clean = F.regexp_replace(tfidf_clean, r'\b\w{1,2}\b', ' ')  # Short words
    tfidf_clean = F.trim(F.regexp_replace(tfidf_clean, r'\s+', ' '))  # Whitespace
    
    # Add both columns in ONE transformation
    result = df.withColumn('text_bert', bert_clean) \
               .withColumn('text_tfidf', tfidf_clean)
    
    print("   ✅ Created 'text_bert' and 'text_tfidf' in SINGLE PASS")
    return result

print("✅ Optimized text cleaning function defined")

## 4. OPTIMIZED Numerical Processing

In [ ]:
# ============================================================
# OPTIMIZED NUMERICAL PROCESSING - Batch Operations
# ============================================================

def optimized_numerical_processing(df, numerical_cols):
    """
    🚀 OPTIMIZED: Batch compute stats and apply transformations
    Instead of multiple passes, compute all stats in ONE aggregation
    """
    print("\n🚀 OPTIMIZED Numerical Processing (batch operations)...")
    
    # ========================================
    # STEP 1: Compute ALL stats in ONE PASS
    # ========================================
    print("\n   Computing statistics in single pass...")
    
    agg_exprs = []
    for col in numerical_cols:
        if col in df.columns:
            agg_exprs.extend([
                F.mean(col).alias(f"{col}_mean"),
                F.stddev(col).alias(f"{col}_stddev"),
                F.expr(f"percentile_approx({col}, 0.5)").alias(f"{col}_median")
            ])
    
    stats = df.select(agg_exprs).collect()[0].asDict()
    
    # ========================================
    # STEP 2: Apply ALL transformations in ONE withColumn chain
    # ========================================
    print("   Applying transformations...")
    
    result = df
    
    for col in numerical_cols:
        if col in df.columns:
            mean = stats[f"{col}_mean"]
            stddev = stats[f"{col}_stddev"]
            median = stats[f"{col}_median"]
            
            # Impute missing with median
            result = result.withColumn(
                col,
                F.coalesce(F.col(col), F.lit(median))
            )
            
            # Add outlier flag (Z-score) in same chain
            if stddev and stddev > 0:
                result = result.withColumn(
                    f"{col}_outlier",
                    F.when(
                        F.abs((F.col(col) - mean) / stddev) > OUTLIER_ZSCORE_THRESHOLD,
                        True
                    ).otherwise(False)
                )
            
            print(f"   ✅ {col}: median={median:.2f}, mean={mean:.2f}, std={stddev:.2f}")
    
    return result

print("✅ Optimized numerical processing function defined")

## 5. Load & Process Review Data (OPTIMIZED)

In [ ]:
# STEP 1: LOAD REVIEW DATA

# Đọc trực tiếp từ Parquet
df_review = spark.read.parquet(REVIEW_RAW_PARQUET)

# 🚀 OPTIMIZATION: Repartition ngay lập tức
df_review = df_review.repartition(NUM_PARTITIONS, "parent_asin", "user_id")

print(f"✅ Loaded {df_review.count():,} reviews from Parquet")

## 6. STEP 2-3: Combined Feature Engineering & Text Cleaning (OPTIMIZED)

In [ ]:
# ============================================================
# 🚀 OPTIMIZED STEPS 2 & 3: Combined in ONE transformation chain
# Instead of multiple passes, we do everything in ONE withColumn chain
# ============================================================

print("\n" + "="*60)
print("STEPS 2-3: FEATURE ENGINEERING & TEXT CLEANING (OPTIMIZED)")
print("="*60)

# 🚀 OPTIMIZATION: Chain ALL transformations together
print("\n🚀 Applying ALL transformations in optimized chain...")

df_review = df_review \
    .withColumn('combined_text',
        F.concat_ws(' ', 
            F.coalesce(F.col('title'), F.lit('')),
            F.coalesce(F.col('text'), F.lit(''))
        )
    ) \
    .withColumn('text_length', F.length(F.col('combined_text'))) \
    .withColumn('review_date', F.from_unixtime(F.col('timestamp')).cast('timestamp')) \
    .withColumn('year', F.year('review_date')) \
    .withColumn('month', F.month('review_date')) \
    .withColumn('day_of_week', F.dayofweek('review_date')) \
    .withColumn('quarter', F.quarter('review_date')) \
    .withColumn('sentiment',
        F.when(F.col('rating') >= 4.0, 'positive')
         .when(F.col('rating') <= 2.0, 'negative')
         .otherwise('neutral')
    ) \
    .withColumn('is_verified',
        F.coalesce(F.col('verified_purchase'), F.lit(False)).cast('integer')
    ) \
    .withColumn('has_helpful_votes',
        F.when(F.col('helpful_vote') > 0, 1).otherwise(0)
    )

print("✅ Basic features created in optimized chain")

# Filter by text length BEFORE text cleaning (saves processing)
print("\n🚀 Filtering by text length BEFORE cleaning...")
before_filter = df_review.count()
df_review = df_review.filter(
    (F.col('text_length') >= MIN_TEXT_LENGTH) & 
    (F.col('text_length') <= MAX_TEXT_LENGTH)
)
after_filter = df_review.count()
print(f"   Filtered: {before_filter:,} -> {after_filter:,} ({100*(before_filter-after_filter)/before_filter:.2f}% removed)")
print("\n🚀 Ghi dữ liệu trung gian ra file Parquet (Checkpointing)...")

temp_path = str(PROCESSED_DIR / "step2_filtered_temp.parquet")
df_review.write.mode("overwrite").parquet(temp_path)
df_review = spark.read.parquet(temp_path)

In [ ]:
# 🚀 OPTIMIZATION: Cache after filtering, before expensive text cleaning
# df_review = df_review.cache()
# df_review.count()  # Trigger cache
print("   ✅ DataFrame cached before text cleaning")

# Apply optimized text cleaning (single pass for both variants)
print("\n🚀 Applying optimized text cleaning...")
df_review = apply_optimized_text_cleaning(df_review)

print("\n✅ Steps 2-3 complete with OPTIMIZATIONS")

## 7. STEP 4: OPTIMIZED Numerical Processing

In [ ]:
# ============================================================
# STEP 4: OPTIMIZED NUMERICAL PROCESSING
# ============================================================

print("\n" + "="*60)
print("STEP 4: NUMERICAL PROCESSING (OPTIMIZED)")
print("="*60)

numerical_cols = ['helpful_vote', 'text_length']

# Apply optimized batch processing
df_review = optimized_numerical_processing(df_review, numerical_cols)

print("\n✅ Step 4 complete with batch optimizations")

## 8. STEP 5: Data Quality & Save (OPTIMIZED)

In [ ]:
# ============================================================
# STEP 5: DATA QUALITY CHECKS (OPTIMIZED)
# ============================================================

print("\n" + "="*60)
print("STEP 5: DATA QUALITY CHECKS (OPTIMIZED)")
print("="*60)

critical_cols = ['user_id', 'parent_asin', 'rating', 'text_bert', 'text_tfidf']

# 🚀 OPTIMIZATION: Check all nulls in ONE pass using array operations
print("\n🚀 Checking nulls in single aggregation...")

null_checks = [F.sum(F.col(c).isNull().cast('int')).alias(f"{c}_nulls") 
               for c in critical_cols if c in df_review.columns]

null_stats = df_review.select(null_checks).collect()[0].asDict()

for col, nulls in null_stats.items():
    print(f"   {col.replace('_nulls', '')}: {nulls:,} nulls")

# Remove nulls
before = df_review.count()
df_review = df_review.na.drop(subset=critical_cols)
after = df_review.count()
print(f"\n   Removed rows with nulls: {before:,} -> {after:,} ({before-after:,} removed)")

# 🚀 OPTIMIZATION: Compute rating and sentiment distribution in ONE aggregation
print("\n🚀 Computing distributions...")
df_review.groupBy('rating', 'sentiment').count() \
    .orderBy('rating').show(15)

print("\n✅ Step 5 complete")

In [ ]:
# ============================================================
# SAVE PROCESSED REVIEW DATA
# ============================================================

print("\n💾 Saving processed review data...")

# 🚀 OPTIMIZATION: Coalesce partitions before writing to reduce small files
df_review.coalesce(NUM_PARTITIONS // 2) \
    .write.mode('overwrite') \
    .parquet(REVIEW_CLEAN_PARQUET)

# Unpersist cache
df_review.unpersist()
print(f"✅ Review data saved to {REVIEW_CLEAN_PARQUET}")
print(f"📊 Final shape: {after:,} rows × {len(df_review.columns)} columns")

## 9. Load & Process Metadata (OPTIMIZED)

In [ ]:
# ============================================================
# STEP 6: LOAD & PROCESS METADATA (OPTIMIZED)
# ============================================================

print("\n" + "="*60)
print("STEP 6: METADATA PROCESSING (OPTIMIZED)")
print("="*60)
df_meta = spark.read.parquet(META_RAW_PARQUET)

print(f"✅ Loaded {df_meta.count():,} metadata records")
# meta_schema = StructType([
#     StructField("parent_asin", StringType(), True),
#     StructField("title", StringType(), True),
#     StructField("price", FloatType(), True),
#     StructField("average_rating", FloatType(), True),
#     StructField("rating_number", IntegerType(), True),
#     StructField("main_category", StringType(), True),
#     StructField("store", StringType(), True),
#     StructField("description", ArrayType(StringType()), True)
# ])


print(f"✅ Loaded {df_meta.count():,} metadata records")

# 🚀 OPTIMIZATION: Chain ALL metadata transformations
print("\n🚀 Processing metadata in optimized chain...")

df_meta = df_meta \
    .withColumn('description_text', F.concat_ws(' ', F.col('description'))) \
    .drop('description') \
    .fillna({
        'price': 0.0,
        'average_rating': 0.0,
        'rating_number': 0,
        'main_category': 'unknown',
        'store': 'unknown',
        'title': 'unknown',
        'description_text': ''
    }) \
    .withColumn('price_category',
        F.when(F.col('price') < 20, 'budget')
         .when(F.col('price') < 50, 'mid-range')
         .when(F.col('price') < 100, 'premium')
         .otherwise('luxury')
    )

print("✅ Metadata processed")

# Save metadata
print(f"\n💾 Saving metadata...")
df_meta.write.mode('overwrite').parquet(META_CLEAN_PARQUET)
print(f"✅ Metadata saved to {META_CLEAN_PARQUET}")

## 10. OPTIMIZED Merge with Broadcast Join

In [ ]:
# # ============================================================
# # STEP 7: MERGE WITH BROADCAST JOIN (MAJOR OPTIMIZATION)
# # ============================================================

# print("\n" + "="*60)
# print("STEP 7: MERGING DATA WITH BROADCAST JOIN (OPTIMIZED)")
# print("="*60)

# # Load data
# df_review = spark.read.parquet(REVIEW_CLEAN_PARQUET)
# df_meta = spark.read.parquet(META_CLEAN_PARQUET)

# df_review = df_review.withColumnRenamed('title', 'review_title')
# df_meta = df_meta.withColumnRenamed('title', 'product_title')
# print(f"\nBefore merge:")
# print(f"   Reviews: {df_review.count():,} rows")
# print(f"   Metadata: {df_meta.count():,} rows")

# # 🚀 MAJOR OPTIMIZATION: Use broadcast join
# # Metadata is smaller, so broadcast it to all workers
# # This avoids expensive shuffle operation!

# print("\n🚀 Using BROADCAST JOIN (no shuffle needed)...")

# df_merged = df_review.join(
#     df_meta,
#     on='parent_asin',
#     how='left'
# )

# # 🚀 OPTIMIZATION: Repartition after join for downstream processing
# df_merged = df_merged.repartition(NUM_PARTITIONS, "parent_asin", "user_id")

# print(f"\n✅ Merge complete with broadcast join (MUCH FASTER!)")
# print(f"   Total rows: {df_merged.count():,}")
# print(f"   Total columns: {len(df_merged.columns)}")

# # Check merge quality
# rows_without_meta = df_merged.filter(F.col('product_title').isNull()).count()
# print(f"Rows without metadata: {rows_without_meta:,}")

# # Save merged data
# print(f"\n💾 Saving merged data...")
# df_merged.write.mode('overwrite').parquet(MERGED_PARQUET)
# print("✅ Merged data saved")

## 11. OPTIMIZED K-Core Filtering

In [3]:
import pyarrow
import sys
print(f"Python path: {sys.executable}")
print(f"PyArrow version: {pyarrow.__version__}")
print(pyarrow.__version__)

Python path: d:\App\anaconda3\envs\amazon_project\python.exe
PyArrow version: 15.0.2
15.0.2


In [10]:
import pandas as pd
import pyspark.sql.functions as F
import gc
import os

print("\n" + "="*60)
print(f"STEP 8: K-CORE FILTERING (K={K_CORE}) - OPTIMIZED HYBRID")
print("="*60)

df_original = spark.read.parquet(MERGED_PARQUET)

# =========================================================================
# BƯỚC 1: LỌC SƠ BỘ TRÊN SPARK (Giảm data xuống 50-70%)
# =========================================================================
print("🔍 Đang lọc sơ bộ trên Spark để giảm kích thước data...")

df_edges = df_original.select('user_id', 'parent_asin').dropDuplicates()

# Lọc users/items có ít hơn K_CORE interactions ngay từ đầu
user_counts = df_edges.groupBy('user_id').count()
item_counts = df_edges.groupBy('parent_asin').count()

df_edges_filtered = (
    df_edges
    .join(user_counts.filter(F.col('count') >= K_CORE).select('user_id'), 'user_id')
    .join(item_counts.filter(F.col('count') >= K_CORE).select('parent_asin'), 'parent_asin')
)

# Kiểm tra kích thước sau lọc
n_before = df_edges.count()
n_after = df_edges_filtered.count()
print(f"📉 Đã giảm từ {n_before:,} → {n_after:,} edges ({(1-n_after/n_before)*100:.1f}% reduction)")

# =========================================================================
# BƯỚC 2: KÉO DATA ĐÃ LỌC VỀ PANDAS
# =========================================================================
print("📥 Đang kéo edge list đã lọc về RAM...")
pdf_edges = df_edges_filtered.toPandas()

print(f"📋 Số lượng tương tác trong RAM: {len(pdf_edges):,} rows")

# =========================================================================
# BƯỚC 3: CHẠY K-CORE TRÊN PANDAS
# =========================================================================
print("\n🚀 Bắt đầu vòng lặp K-Core trên RAM...")
iteration = 0
while True:
    iteration += 1
    n_before = len(pdf_edges)

    # Lọc items
    item_counts = pdf_edges['parent_asin'].value_counts()
    valid_items = item_counts[item_counts >= K_CORE].index
    pdf_edges = pdf_edges[pdf_edges['parent_asin'].isin(valid_items)]

    # Lọc users
    user_counts = pdf_edges['user_id'].value_counts()
    valid_users = user_counts[user_counts >= K_CORE].index
    pdf_edges = pdf_edges[pdf_edges['user_id'].isin(valid_users)]

    n_after = len(pdf_edges)
    removed = n_before - n_after
    
    print(f"🔄 Vòng {iteration:2d}: {n_before:>12,} → {n_after:>12,} (xóa {removed:>8,})")

    if removed == 0:
        print(f"\n✅ Hội tụ sau {iteration} vòng lặp!")
        break

# =========================================================================
# BƯỚC 4: ĐƯA KẾT QUẢ TRỞ LẠI SPARK
# =========================================================================
print("\n📤 Đang lưu danh sách ID hợp lệ...")

valid_users_list = pdf_edges['user_id'].unique()
valid_items_list = pdf_edges['parent_asin'].unique()

# Lưu qua Parquet (tránh lỗi createDataFrame với data lớn)
pd.DataFrame({'user_id': valid_users_list}).to_parquet("temp_valid_users.parquet", index=False)
pd.DataFrame({'parent_asin': valid_items_list}).to_parquet("temp_valid_items.parquet", index=False)

del pdf_edges, valid_users_list, valid_items_list
gc.collect()

sdf_valid_users = spark.read.parquet("temp_valid_users.parquet")
sdf_valid_items = spark.read.parquet("temp_valid_items.parquet")

# =========================================================================
# BƯỚC 5: JOIN VỚI DATA GỐC
# =========================================================================
print("🔄 Đang khôi phục metadata...")

df_final = (
    df_original
    .join(sdf_valid_users, on='user_id', how='inner')
    .join(sdf_valid_items, on='parent_asin', how='inner')
)

df_final.cache()

n_interactions = df_final.count()
n_users = sdf_valid_users.count()
n_items = sdf_valid_items.count()
density = n_interactions / (n_users * n_items) * 100

print(f"\n📊 Thống kê K-Core ({K_CORE}):")
print(f"   Tổng tương tác : {n_interactions:,}")
print(f"   Unique users   : {n_users:,}")
print(f"   Unique items   : {n_items:,}")
print(f"   Density        : {density:.4f}%")

# Lưu kết quả
# df_final.write.mode('overwrite').parquet(FINAL_PARQUET)

print("\n🧹 Hoàn thành!")


STEP 8: K-CORE FILTERING (K=5) - OPTIMIZED HYBRID
🔍 Đang lọc sơ bộ trên Spark để giảm kích thước data...
📉 Đã giảm từ 64,908,952 → 27,675,255 edges (57.4% reduction)
📥 Đang kéo edge list đã lọc về RAM...
📋 Số lượng tương tác trong RAM: 27,675,255 rows

🚀 Bắt đầu vòng lặp K-Core trên RAM...
🔄 Vòng  1:   27,675,255 →   23,451,319 (xóa 4,223,936)
🔄 Vòng  2:   23,451,319 →   22,975,503 (xóa  475,816)
🔄 Vòng  3:   22,975,503 →   22,940,210 (xóa   35,293)
🔄 Vòng  4:   22,940,210 →   22,937,404 (xóa    2,806)
🔄 Vòng  5:   22,937,404 →   22,937,180 (xóa      224)
🔄 Vòng  6:   22,937,180 →   22,937,152 (xóa       28)
🔄 Vòng  7:   22,937,152 →   22,937,152 (xóa        0)

✅ Hội tụ sau 7 vòng lặp!

📤 Đang lưu danh sách ID hợp lệ...
🔄 Đang khôi phục metadata...

📊 Thống kê K-Core (5):
   Tổng tương tác : 23,202,904
   Unique users   : 2,509,926
   Unique items   : 711,427
   Density        : 0.0013%

🧹 Hoàn thành!


## 12. Save Final Dataset

In [11]:
import math

print("\n" + "="*60)
print("STEP 9: SAVE FINAL GOLD DATASET")
print("="*60)

# =========================================================================
# TÍNH TOÁN SỐ LƯỢNG FILE ĐẦU RA TỐI ƯU
# Quy tắc Data Engineering: ~ 1-2 triệu dòng cho 1 partition (hoặc dựa theo Core máy)
# =========================================================================
optimal_partitions = max(1, min(safe_cores, math.ceil(n_interactions / 1500000)))

print(f"\n💾 Đang lưu bộ dữ liệu Gold (gom thành {optimal_partitions} file Parquet)...")

try:
    (
        df_final.coalesce(optimal_partitions)
        .write.mode('overwrite')
        .parquet(FINAL_PARQUET)
    )

    print(f"\n✨ HOÀN THÀNH XUẤT SẮC! Dữ liệu đã được lưu an toàn tại:\n   📁 {FINAL_PARQUET}")

    print(f"\n📊 THỐNG KÊ DATASET CUỐI CÙNG:")
    print(f"   Tổng tương tác : {n_interactions:>10,}")
    print(f"   Số lượng Users : {n_users:>10,}")
    print(f"   Số lượng Items : {n_items:>10,}")

    
finally:
    os.remove("temp_valid_users.parquet")
    os.remove("temp_valid_items.parquet")
    # Bắt buộc giải phóng RAM sau khi lưu xong
    df_final.unpersist()
    print("\n🧹 Đã giải phóng RAM cho df_final.")


STEP 9: SAVE FINAL GOLD DATASET

💾 Đang lưu bộ dữ liệu Gold (gom thành 9 file Parquet)...

✨ HOÀN THÀNH XUẤT SẮC! Dữ liệu đã được lưu an toàn tại:
   📁 D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\amazon_final_gold_spark.parquet

📊 THỐNG KÊ DATASET CUỐI CÙNG:
   Tổng tương tác : 23,202,904
   Số lượng Users :  2,509,926
   Số lượng Items :    711,427

🧹 Đã giải phóng RAM cho df_final.


In [ ]:
# Stop Spark (uncomment to use)
spark.stop()
# print("✅ Spark session stopped")